# Inference demo — ตัวอักษรไทย 72 คลาส

ใช้ DenseNet121 E1 candidate ที่อยู่ใน repo โดยเรียก `src.inference` ชุดเดียวกับคำสั่ง CLI ไม่ต้องเทรนใหม่

1. ติดตั้ง `requirements-training.txt` และเลือก kernel `.venv`
2. แก้ `INPUT_PATH` ใน cell ถัดไปเป็นรูปหนึ่งไฟล์หรือโฟลเดอร์รูป (path เริ่มจาก root ของ repo หรือ absolute path)
3. กด **Run All**; รูปเดี่ยวแสดง Top-3 ใน notebook ส่วนโฟลเดอร์บันทึก CSV

โมเดลนี้รับภาพตัวอักษรเดี่ยวหนึ่งตัวต่อภาพ; ยังไม่ใช่ OCR สำหรับข้อความทั้งบรรทัด และยังไม่ใช่โมเดลที่เลือกหลังเทียบ E1/E2

In [ ]:
INPUT_PATH = ""  # เช่น images/character.png หรือ images/characters/
DEVICE = "cpu"  # เปลี่ยนเป็น auto เพื่อเลือก CUDA/MPS เมื่อมี
TOP_K = 3
OUTPUT_CSV = "results/predictions/notebook_predictions.csv"

In [ ]:
from pathlib import Path
import sys
from IPython.display import display
from PIL import Image

repo_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/inference.py").is_file()), None)
if repo_root is None:
    raise FileNotFoundError("เปิด notebook จากภายในโฟลเดอร์ repo")
sys.path.insert(0, str(repo_root))
from src.inference import Predictor, discover_defaults, discover_images, print_single, save_csv

weights, config, labels = discover_defaults()
predictor = Predictor(weights, device=DEVICE, config_path=config, labels_path=labels)
print(f"พร้อมทำนาย: {predictor.config.architecture}, {len(predictor.labels)} คลาส, device={predictor.device}")

In [ ]:
if not INPUT_PATH:
    print("ตั้ง INPUT_PATH ใน cell แรกเป็นไฟล์รูปหรือโฟลเดอร์รูป แล้วรัน cell นี้อีกครั้ง")
else:
    input_path = Path(INPUT_PATH).expanduser()
    if not input_path.is_absolute():
        input_path = repo_root / input_path
    image_paths = discover_images(input_path)
    if not image_paths:
        raise ValueError(f"ไม่พบรูปใน {input_path}")
    predictions = predictor.predict(image_paths, top_k=TOP_K)
    if input_path.is_file():
        with Image.open(input_path) as image:
            display(image.copy())
        print_single(predictions[0], predictor, input_path, "none")
    else:
        csv_path = repo_root / OUTPUT_CSV
        save_csv(csv_path, predictions)
        print(f"ทำนาย {len(predictions)} ภาพแล้ว: {csv_path}")
        for row in predictions[:5]:
            print(f"{row['filename']}: {row['prediction']} ({row['status']})")